In [1]:
pip install ultralytics


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install opencv-python


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install torch


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install torchvision


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install torchaudio


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import glob
import xml.etree.ElementTree as ET

# Define class mappings based on dataset labels
class_mapping = {
    "stop": 0,
    "speedlimit": 1,
    "crosswalk": 2,
    "trafficlight": 3
}

# Paths (update these)
xml_folder = "archive-2/annotations"  # XML files folder
output_folder = "archive-2/labels"  # Output TXT folder
os.makedirs(output_folder, exist_ok=True)

for xml_file in glob.glob(os.path.join(xml_folder, "*.xml")):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    img_width = int(root.find("size/width").text)
    img_height = int(root.find("size/height").text)

    txt_filename = os.path.join(output_folder, os.path.basename(xml_file).replace(".xml", ".txt"))
    object_count = 0  # Track if any objects are written

    print(f"Processing {xml_file}...")  # Debug

    with open(txt_filename, "w") as txt_file:
        for obj in root.findall("object"):
            class_name = obj.find("name").text.lower()
            print(f"Found class: {class_name}")  # Debug

            if class_name not in class_mapping:
                print(f"Skipping unknown class: {class_name}")  # Debug
                continue  # Skip unknown labels

            class_id = class_mapping[class_name]
            bbox = obj.find("bndbox")

            try:
                xmin = int(bbox.find("xmin").text)
                ymin = int(bbox.find("ymin").text)
                xmax = int(bbox.find("xmax").text)
                ymax = int(bbox.find("ymax").text)

                # Convert to YOLO format (normalize values)
                x_center = ((xmin + xmax) / 2) / img_width
                y_center = ((ymin + ymax) / 2) / img_height
                width = (xmax - xmin) / img_width
                height = (ymax - ymin) / img_height

                print(f"Object {class_name}: {xmin}, {ymin}, {xmax}, {ymax} -> ({x_center:.6f}, {y_center:.6f}, {width:.6f}, {height:.6f})")  # Debug

                # Write to YOLO format .txt file
                txt_file.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
                object_count += 1
            except Exception as e:
                print(f"Error processing {xml_file}: {e}")

    if object_count == 0:
        print(f"Warning: No objects found in {xml_file}!")

print("Conversion complete! Check label files in:", output_folder)

Processing archive-2/annotations/road712.xml...
Found class: speedlimit
Object speedlimit: 98, 140, 139, 182 -> (0.395000, 0.402500, 0.136667, 0.105000)
Found class: speedlimit
Object speedlimit: 97, 205, 138, 246 -> (0.391667, 0.563750, 0.136667, 0.102500)
Processing archive-2/annotations/road706.xml...
Found class: speedlimit
Object speedlimit: 136, 92, 177, 135 -> (0.521667, 0.283750, 0.136667, 0.107500)
Found class: speedlimit
Object speedlimit: 135, 159, 177, 201 -> (0.520000, 0.450000, 0.140000, 0.105000)
Processing archive-2/annotations/road289.xml...
Found class: stop
Object stop: 61, 140, 146, 227 -> (0.345000, 0.458750, 0.283333, 0.217500)
Found class: trafficlight
Object trafficlight: 259, 325, 299, 398 -> (0.930000, 0.903750, 0.133333, 0.182500)
Processing archive-2/annotations/road538.xml...
Found class: speedlimit
Object speedlimit: 115, 169, 149, 205 -> (0.440000, 0.467500, 0.113333, 0.090000)
Processing archive-2/annotations/road510.xml...
Found class: speedlimit
Object

In [18]:
import torch
print(torch.backends.mps.is_available())  # Should print True

True


In [20]:
from ultralytics import YOLO

# Load the model
model = YOLO("garr8n.pt")

# Train the model
model.train(data="/Users/kalpit//611/road_sign_database/dataset.yaml", epochs=10, imgsz=640, batch=16)

New https://pypi.org/project/ultralytics/8.3.82 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.81 🚀 Python-3.12.6 torch-2.6.0 CPU (Apple M3)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/Users/kalpit//611/road_sign_database/dataset.yaml, epochs=30, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train20, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_f

train: Scanning /Users/kalpit/611/road_sign_database/train/labels.cache... 600 i
val: Scanning /Users/kalpit/611/road_sign_database/val/labels.cache... 277 image

Plotting labels to /opt/homebrew/runs/detect/train20/labels.jpg... 


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.00125, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /opt/homebrew/runs/detect/train20
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/30         0G     0.8153      2.649     0.9882         19        640: 1
                 Class     Images  Instances      Box(P          R      mAP50  m

...

                   all        277        440      0.888      0.818       0.87      0.713



30 epochs completed in 2.078 hours.
Optimizer stripped from /opt/homebrew/runs/detect/train20/weights/last.pt, 6.2MB
Optimizer stripped from /opt/homebrew/runs/detect/train20/weights/best.pt, 6.2MB

Validating /opt/homebrew/runs/detect/train20/weights/best.pt...
Ultralytics 8.3.81 🚀 Python-3.12.6 torch-2.6.0 CPU (Apple M3)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  m


                   all        277        440      0.888      0.818       0.87      0.713
             stop_sign         25         25      0.903       0.72       0.87      0.769
           speed_limit        258        336      0.933      0.985       0.99      0.893
             crosswalk         40         46      0.927      0.891      0.908      0.715
         traffic_light         11         33      0.788      0.676      0.713      0.473
Speed: 0.7ms preprocess, 94.4ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /opt/homebrew/runs/detect/train20


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x16bb0d070>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [1]:
# Run inference on the video
results = model("main1.MP4", save=True, stream=True)  # Saves the processed video

NameError: name 'model' is not defined

In [4]:
from ultralytics import YOLO

# Load the trained model (best.pt)
model = YOLO("p2/M18n.pt")

# Test the model on a single image
# results = model("path_t.jpg")  # Provide the path to your image
# results.show()  # Show the results

# Test the model on a video
results = model("main3.mp4", conf=0.5, save=True, stream=True)  # Process video, save output, and stream results
for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs
# Iterate over the results generator


video 1/1 (frame 1/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 30.4ms
video 1/1 (frame 2/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 35.7ms
video 1/1 (frame 3/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 2 trucks, 31.3ms
video 1/1 (frame 4/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 32.4ms
video 1/1 (frame 5/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 30.6ms
video 1/1 (frame 6/5412) /Users/kalpit/611/main3.mp4: 384x640 2 cars, 1 truck, 28.3ms
video 1/1 (frame 7/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 28.2ms
video 1/1 (frame 8/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 33.0ms
video 1/1 (frame 9/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 31.1ms
video 1/1 (frame 10/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 29.5ms
video 1/1 (frame 11/5412) /Users/kalpit/611/main3.mp4: 384x640 2 cars, 32.2ms
video 1/1 (frame 12/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 30.4ms
video 1/1

In [5]:
from ultralytics import YOLO

# Load the trained model (best.pt)
model = YOLO("p2/M28n.pt")

# Test the model on a single image
# results = model("path_t.jpg")  # Provide the path to your image
# results.show()  # Show the results

# Test the model on a video
results = model("main3.mp4", conf=0.5, save=True, stream=True)  # Process video, save output, and stream results
for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs
# Iterate over the results generator


video 1/1 (frame 1/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 34.4ms
video 1/1 (frame 2/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 31.6ms
video 1/1 (frame 3/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 2 trucks, 28.8ms
video 1/1 (frame 4/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 29.3ms
video 1/1 (frame 5/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 29.6ms
video 1/1 (frame 6/5412) /Users/kalpit/611/main3.mp4: 384x640 2 cars, 1 truck, 32.2ms
video 1/1 (frame 7/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 34.1ms
video 1/1 (frame 8/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 27.0ms
video 1/1 (frame 9/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 26.6ms
video 1/1 (frame 10/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 34.0ms
video 1/1 (frame 11/5412) /Users/kalpit/611/main3.mp4: 384x640 2 cars, 30.4ms
video 1/1 (frame 12/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 28.9ms
video 1/1

In [6]:
from ultralytics import YOLO

# Load the trained model (best.pt)
model = YOLO("p2/M38n.pt")

# Test the model on a single image
# results = model("path_t.jpg")  # Provide the path to your image
# results.show()  # Show the results

# Test the model on a video
results = model("main3.mp4", conf=0.5, save=True, stream=True)  # Process video, save output, and stream results
for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs
# Iterate over the results generator


video 1/1 (frame 1/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 47.8ms
video 1/1 (frame 2/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 34.1ms
video 1/1 (frame 3/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 2 trucks, 35.3ms
video 1/1 (frame 4/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 33.7ms
video 1/1 (frame 5/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 30.1ms
video 1/1 (frame 6/5412) /Users/kalpit/611/main3.mp4: 384x640 2 cars, 1 truck, 30.3ms
video 1/1 (frame 7/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 33.5ms
video 1/1 (frame 8/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 28.9ms
video 1/1 (frame 9/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 30.5ms
video 1/1 (frame 10/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 31.6ms
video 1/1 (frame 11/5412) /Users/kalpit/611/main3.mp4: 384x640 2 cars, 30.2ms
video 1/1 (frame 12/5412) /Users/kalpit/611/main3.mp4: 384x640 1 car, 1 truck, 35.5ms
video 1/1